# Credit Risk Prediction Model — Revised Version 7

Perbaikan dari v6:

1. **Penamaan variabel & penjelasan `emp_length` diperbaiki** — sebelumnya dikelompokkan sebagai "kardinalitas tinggi" bersama `issue_d`/`earliest_cr_line`, padahal `emp_length` hanya punya sekitar 11 kategori (`< 1 year` s.d. `10+ years`). Penjelasannya sekarang dipisah sesuai alasan masing-masing.
2. **Redaksi bagian `class_weight='balanced'` diperjelas** — kalimat "model berisiko malas mengenali kelas minoritas" diganti dengan penjelasan yang lebih formal.
3. **`max_depth` Random Forest dibatasi** — hasil v6 menunjukkan Random Forest overfitting parah (ROC AUC training 0.9998 vs test 0.698). Grid parameter diperketat untuk membatasi kedalaman pohon, sebagai upaya mengurangi overfitting tersebut.

Semua perbaikan dari v5 (capping outlier, kolom leakage dibuang, target mapping, imputer di dalam Pipeline, ROC AUC dari probability) tetap dipertahankan, karena sudah benar.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

## 1. Data Collection

In [4]:
from google.colab import drive
drive.mount('/content/drive')

data = pd.read_csv('/content/drive/MyDrive/loan_data_2007_2014.csv')

# Membuat DataFrame
df = pd.DataFrame(data)

# Cek data
df.head()

Mounted at /content/drive


,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Membuang Kolom Leakage & Kolom Tidak Relevan

Kolom-kolom berikut dibuang karena nilainya baru terisi SETELAH status pinjaman final diketahui (post-outcome), sehingga memakainya sebagai fitur = memberi tahu model jawabannya duluan:

- `total_pymnt`, `total_pymnt_inv`, `total_rec_prncp`, `total_rec_int`, `total_rec_late_fee` — jumlah yang sudah dibayar/ditagih
- `recoveries`, `collection_recovery_fee` — dana yang ditagih SETELAH gagal bayar
- `out_prncp`, `out_prncp_inv` — sisa pokok pinjaman
- `last_pymnt_d`, `last_pymnt_amnt`, `next_pymnt_d`, `last_credit_pull_d` — tanggal/jumlah pembayaran terakhir

Ditambah kolom ID/teks bebas yang tidak informatif untuk prediksi (`Unnamed: 0`, `id`, `member_id`, `url`, `desc`, `title`, `zip_code`, `emp_title`, `policy_code`).

Serta kolom `issue_d` dan `earliest_cr_line` — keduanya bertipe object berisi tanggal dengan banyak nilai unik (terutama `earliest_cr_line`, yang bisa merentang puluhan tahun ke belakang), sehingga berisiko menghasilkan terlalu banyak kolom dummy saat di-*one-hot encode* kalau tidak dibuang. `emp_length` dibuang terpisah untuk menyederhanakan fitur yang digunakan dalam pemodelan (bukan karena kardinalitas tinggi — nilainya hanya sekitar 11 kategori).

In [5]:
leakage_cols = [
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_prncp',
    'total_rec_int',
    'total_rec_late_fee',
    'recoveries',
    'collection_recovery_fee',
    'out_prncp',
    'out_prncp_inv',
    'last_pymnt_d',
    'last_pymnt_amnt',
    'next_pymnt_d',
    'last_credit_pull_d',
]

non_informative_cols = [
    'Unnamed: 0',
    'id',
    'member_id',
    'url',
    'desc',
    'title',
    'zip_code',
    'emp_title',
    'policy_code',
]

# Kolom tanggal ber-kardinalitas tinggi -- bertipe object sehingga akan
# otomatis masuk categorical_cols dan di-OneHotEncode kalau tidak dibuang.
# earliest_cr_line terutama bisa punya ratusan nilai unik (riwayat kredit
# peminjam bisa dimulai puluhan tahun sebelum data ini dikumpulkan).
high_cardinality_date_cols = [
    'issue_d',
    'earliest_cr_line',
]

# emp_length dibuang terpisah untuk menyederhanakan fitur -- BUKAN karena
# kardinalitas tinggi (nilainya hanya sekitar 11 kategori: '< 1 year' s.d. '10+ years').
simplification_cols = [
    'emp_length',
]

cols_to_drop = [
    c for c in leakage_cols + non_informative_cols + high_cardinality_date_cols + simplification_cols
    if c in df.columns
]

df = df.drop(columns=cols_to_drop)

print(f"Kolom yang dibuang ({len(cols_to_drop)}):")
print(cols_to_drop)

print()
print(f"Jumlah kolom tersisa: {df.shape[1]}")

Kolom yang dibuang (25):
['total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'out_prncp', 'out_prncp_inv', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'Unnamed: 0', 'id', 'member_id', 'url', 'desc', 'title', 'zip_code', 'emp_title', 'policy_code', 'issue_d', 'earliest_cr_line', 'emp_length']

Jumlah kolom tersisa: 50


## 3. Membersihkan & Memfinalisasi Target (`loan_status`)

Hanya pinjaman dengan status FINAL (`Fully Paid` / `Charged Off` / `Default`) yang dipakai. Status yang belum final (`Current`, `Late`, `In Grace Period`, dll) dibuang, bukan otomatis dianggap `GOOD`.

In [6]:
print("Distribusi loan_status sebelum difilter:")
print(df['loan_status'].value_counts())

status_map = {
    'Fully Paid': 1,
    'Does not meet the credit policy. Status:Fully Paid': 1,
    'Charged Off': 0,
    'Does not meet the credit policy. Status:Charged Off': 0,
    'Default': 0,
}

df = df[df['loan_status'].isin(status_map.keys())].copy()
df['loan_status'] = df['loan_status'].map(status_map)

print()
print(f"Jumlah baris setelah hanya mengambil status final: {len(df)}")

print()
print("Distribusi loan_status setelah difilter:")
print(df['loan_status'].value_counts())

Distribusi loan_status sebelum difilter:
loan_status
Fully Paid                                             83905
Current                                                60877
Charged Off                                            19488
Does not meet the credit policy. Status:Fully Paid      1988
Late (31-120 days)                                      1766
In Grace Period                                          948
Does not meet the credit policy. Status:Charged Off      761
Late (16-30 days)                                        316
Default                                                  180
Name: count, dtype: int64

Jumlah baris setelah hanya mengambil status final: 106322

Distribusi loan_status setelah difilter:
loan_status
1    85893
0    20429
Name: count, dtype: int64


## 4. Identifikasi Label & Fitur

In [7]:
label_col = 'loan_status'

df = df.dropna(thresh=len(df) * 0.5, axis=1)

X = df.drop(columns=[label_col])
y = df[label_col]

## 5. Train-Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Copy eksplisit untuk menghindari SettingWithCopyWarning saat clipping/assignment
# di langkah-langkah berikutnya.
X_train = X_train.copy()
X_test = X_test.copy()

numeric_cols = X_train.select_dtypes(include=[np.number]).columns
categorical_cols = X_train.select_dtypes(include=[object]).columns

## 6. Penanganan Outlier — Capping / Winsorization

**Perubahan utama di versi ini.** Sebelumnya, baris dibuang kalau ADA SAJA satu kolom numerik yang di luar batas IQR. Karena dataset ini punya puluhan kolom numerik yang secara alami skewed (income, saldo, dll), aturan itu bisa membuang mayoritas baris training — peluang satu baris "aman" di semua kolom sekaligus turun drastis seiring bertambahnya jumlah kolom.

Sekarang, nilai yang melewati batas IQR **dipotong (clip)** ke batas atas/bawah, bukan barisnya yang dibuang:

- Batas IQR tetap dihitung HANYA dari `X_train` (tidak ada perubahan di sini, masih no-leakage).
- Batas yang sama itu diterapkan ke `X_train` MAUPUN `X_test` lewat `.clip()` — bukan menghitung ulang IQR dari `X_test`.
- Tidak ada baris yang hilang dari `X_train` maupun `X_test`.

In [9]:
iqr_bounds = {}

for col in numeric_cols:
    col_train = X_train[col].dropna()

    Q1 = col_train.quantile(0.25)
    Q3 = col_train.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    iqr_bounds[col] = (lower_bound, upper_bound)

for col, (lower_bound, upper_bound) in iqr_bounds.items():
    X_train[col] = X_train[col].clip(lower=lower_bound, upper=upper_bound)
    X_test[col] = X_test[col].clip(lower=lower_bound, upper=upper_bound)

print(f"Jumlah baris X_train: {len(X_train)} (tidak ada yang dibuang)")
print(f"Jumlah baris X_test: {len(X_test)} (tidak ada yang dibuang)")

print()
print("Contoh batas capping untuk 5 kolom pertama:")
for col in list(iqr_bounds.keys())[:5]:
    lower_bound, upper_bound = iqr_bounds[col]
    print(f"  {col}: [{lower_bound:.2f}, {upper_bound:.2f}]")

Jumlah baris X_train: 85057 (tidak ada yang dibuang)
Jumlah baris X_test: 21265 (tidak ada yang dibuang)

Contoh batas capping untuk 5 kolom pertama:
  loan_amnt: [-9500.00, 34500.00]
  funded_amnt: [-9500.00, 34500.00]
  funded_amnt_inv: [-10187.50, 34112.50]
  int_rate: [2.12, 24.84]
  installment: [-240.48, 987.12]


## 7. Preprocessing Pipeline (Imputer + Scaling + Encoding)

`SimpleImputer` tetap jadi bagian dari `ColumnTransformer`/Pipeline, di-*fit* ulang di setiap fold cross-validation.

In [10]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

## 8. Modelling & Evaluation

In [11]:
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, solver='lbfgs', class_weight='balanced')),
])

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42, class_weight='balanced')),
])

param_grid_lr = {
    'model__C': [0.01, 0.1, 1, 10],
}

grid_search_lr = GridSearchCV(
    pipeline_lr,
    param_grid_lr,
    cv=3,
    scoring='roc_auc',
)
grid_search_lr.fit(X_train, y_train)

# max_depth dibatasi (dibandingkan v6 yang menyertakan None / kedalaman tak terbatas)
# untuk mengurangi overfitting yang terlihat di hasil v6
# (ROC AUC training 0.9998 vs test 0.698).
param_grid_rf = {
    'model__n_estimators': [100],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_split': [10, 20],
    'model__min_samples_leaf': [5, 10],
}

grid_search_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=3,
    scoring='roc_auc',
)
grid_search_rf.fit(X_train, y_train)

models = {
    'Logistic Regression': grid_search_lr,
    'Random Forest': grid_search_rf,
}

## 9. Evaluasi Hasil

In [12]:
model_results = {
    'Model': [],
    'Dataset': [],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'ROC AUC': [],
}

for model_name, model in models.items():
    datasets = [
        ('Training', (X_train, y_train)),
        ('Test', (X_test, y_test)),
    ]

    for dataset_name, (X_eval, y_eval) in datasets:
        y_pred = model.predict(X_eval)
        y_prob = model.predict_proba(X_eval)[:, 1]

        model_results['Model'].append(model_name)
        model_results['Dataset'].append(dataset_name)
        model_results['Accuracy'].append(accuracy_score(y_eval, y_pred))
        model_results['Precision'].append(precision_score(y_eval, y_pred))
        model_results['Recall'].append(recall_score(y_eval, y_pred))
        model_results['ROC AUC'].append(roc_auc_score(y_eval, y_prob))

results_df = pd.DataFrame(model_results)
print(results_df)

                 Model   Dataset  Accuracy  Precision    Recall   ROC AUC
0  Logistic Regression  Training  0.642275   0.891811  0.634121  0.709182
1  Logistic Regression      Test  0.640301   0.891537  0.631585  0.708512
2        Random Forest  Training  0.720082   0.913208  0.722138  0.794043
3        Random Forest      Test  0.675100   0.880201  0.692008  0.703073


In [13]:
for model_name, model in models.items():
    y_pred_test = model.predict(X_test)

    print(f"Model: {model_name}")
    print()

    print("Classification Report:")
    print(classification_report(y_test, y_pred_test))

    cm = confusion_matrix(y_test, y_pred_test)
    print("Confusion Matrix:")
    print(cm)

    print("-" * 80)

print()
print("Modelling selesai (v7: outlier di-capping bukan dibuang, tanpa feature leakage, tanpa CV leakage, ROC AUC dari probability).")

Model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.30      0.68      0.42      4086
           1       0.89      0.63      0.74     17179

    accuracy                           0.64     21265
   macro avg       0.60      0.65      0.58     21265
weighted avg       0.78      0.64      0.68     21265

Confusion Matrix:
[[ 2766  1320]
 [ 6329 10850]]
--------------------------------------------------------------------------------
Model: Random Forest

Classification Report:
              precision    recall  f1-score   support

           0       0.32      0.60      0.42      4086
           1       0.88      0.69      0.77     17179

    accuracy                           0.68     21265
   macro avg       0.60      0.65      0.60     21265
weighted avg       0.77      0.68      0.71     21265

Confusion Matrix:
[[ 2468  1618]
 [ 5291 11888]]
-----------------------------------------------------------------------